# SeraTune TRIBE v2 Worker

Run all cells top to bottom. No restarts needed.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU** (or A100 with Colab Pro)
2. Have your [HuggingFace token](https://huggingface.co/settings/tokens) ready (accept [LLaMA 3.2-3B license](https://huggingface.co/meta-llama/Llama-3.2-3B) first)
3. Have your [ngrok auth token](https://dashboard.ngrok.com/get-started/your-authtoken) ready

**After all cells run**, copy the ngrok URL into your backend `.env`:
```
TRIBE_WORKER_URL=https://xxxx.ngrok-free.app
USE_MOCK_TRIBE=false
```

In [ ]:
# ============================================================
# STEP 1: Install everything (takes ~5 min, no restart needed)
# ============================================================
import subprocess, sys

# Uninstall conflicting Colab packages silently
for pkg in ['torch', 'torchaudio', 'torchvision', 'numpy',
            'neuralset', 'neuraltrain', 'tribev2', 'click', 'gtts']:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg],
                   capture_output=True)

# Purge old modules from memory so fresh installs are used
import importlib
mods_to_purge = [k for k in sys.modules if k.startswith(('numpy', 'torch', 'tribev2'))]
for m in mods_to_purge:
    del sys.modules[m]

# Install pinned compatible versions
!pip install -q \
  "numpy==2.2.6" \
  "torch==2.6.0" \
  "torchaudio==2.6.0" \
  "torchvision==0.21.0" \
  --index-url https://download.pytorch.org/whl/cu124

!pip install -q \
  "neuralset==0.0.2" \
  "neuraltrain==0.0.2" \
  "x_transformers==1.27.20" \
  "moviepy>=2.2.1" \
  gtts julius Levenshtein transformers huggingface_hub \
  soundfile librosa langdetect spacy

!pip install -q git+https://github.com/facebookresearch/tribev2.git
!pip install -q fastapi uvicorn pyngrok python-multipart

print('\n✅ All packages installed! Continue to the next cell.')

In [ ]:
# ============================================================
# STEP 2: Authenticate with HuggingFace
# ============================================================
from huggingface_hub import login

HF_TOKEN = ""  # paste your token here, or leave empty to be prompted

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()

In [ ]:
# ============================================================
# STEP 3: Load TRIBE v2 model
# ============================================================
from tribev2 import TribeModel
import torch

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\nLoading TRIBE v2 model...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='./cache')
print('✅ Model loaded!')

In [ ]:
# ============================================================
# STEP 4: Quick inference test
# ============================================================
import numpy as np, wave

sr = 16000
t = np.linspace(0, 5, sr * 5, endpoint=False)
audio = (16000 * np.sin(2 * np.pi * 440 * t)).astype(np.int16)

with wave.open('/tmp/test_tone.wav', 'w') as wf:
    wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
    wf.writeframes(audio.tobytes())

df = model.get_events_dataframe(audio_path='/tmp/test_tone.wav')
preds, segments = model.predict(events=df)
print(f'Predictions: {preds.shape}  ({preds.shape[0]} segments × {preds.shape[1]} vertices)')
print('✅ Inference test passed!')

In [ ]:
%%writefile /tmp/tribe_worker.py
"""TRIBE v2 inference worker — lightweight FastAPI server."""

import logging, os, tempfile, time
import numpy as np
from fastapi import FastAPI, File, HTTPException, UploadFile
from tribev2 import TribeModel

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title='TRIBE v2 Worker')
_model: TribeModel | None = None

def get_model() -> TribeModel:
    global _model
    if _model is None:
        logger.info('Loading TRIBE v2 model...')
        _model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='./cache')
        logger.info('Model loaded.')
    return _model

@app.on_event('startup')
async def startup(): get_model()

@app.get('/health')
async def health(): return {'status': 'ok', 'model_loaded': _model is not None}

@app.post('/analyze')
async def analyze(audio: UploadFile = File(...)):
    model = get_model()
    tmp_path = None
    try:
        suffix = os.path.splitext(audio.filename or 'audio.wav')[1] or '.wav'
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp_path = tmp.name
            content = await audio.read()
            tmp.write(content)
        logger.info('Running inference on %s (%d bytes)...', audio.filename, len(content))
        start = time.time()
        df = model.get_events_dataframe(audio_path=tmp_path)
        preds, segments = model.predict(events=df)
        elapsed = time.time() - start
        logger.info('Inference complete: %s → %s in %.1fs', audio.filename, preds.shape, elapsed)
        return {
            'preds': preds.tolist(),
            'shape': list(preds.shape),
            'n_segments': preds.shape[0],
            'n_vertices': preds.shape[1],
            'inference_time_s': round(elapsed, 2),
        }
    except Exception as e:
        logger.exception('Inference failed')
        raise HTTPException(500, f'Inference failed: {e}')
    finally:
        if tmp_path: os.unlink(tmp_path)

In [ ]:
# ============================================================
# STEP 6: Enter your ngrok token + start the server
# ============================================================
import threading, uvicorn, importlib.util
from pyngrok import ngrok

NGROK_AUTH_TOKEN = ""  # paste your ngrok auth token here
if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = input('Enter your ngrok auth token: ')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Load the worker module and inject the already-loaded model
spec = importlib.util.spec_from_file_location('tribe_worker', '/tmp/tribe_worker.py')
tribe_worker = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tribe_worker)
tribe_worker._model = model  # reuse model from step 3

# Start uvicorn in a background thread
config = uvicorn.Config(tribe_worker.app, host='0.0.0.0', port=8001, log_level='info')
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

import time; time.sleep(2)  # wait for server to start

# Open ngrok tunnel
public_url = ngrok.connect(8001).public_url

print('\n' + '='*60)
print(f'✅ TRIBE v2 Worker is running!')
print(f'\n   Public URL: {public_url}')
print(f'\n   Add this to your backend .env:')
print(f'     TRIBE_WORKER_URL={public_url}')
print(f'     USE_MOCK_TRIBE=false')
print(f'\n   Health check: {public_url}/health')
print('='*60)

In [ ]:
# ============================================================
# STEP 7: Test the worker endpoint
# ============================================================
import httpx

# Test health
r = httpx.get(f'{public_url}/health', headers={'ngrok-skip-browser-warning': 'true'})
print(f'Health: {r.json()}')

# Test inference via HTTP
with open('/tmp/test_tone.wav', 'rb') as f:
    r = httpx.post(f'{public_url}/analyze',
                   files={'audio': ('test.wav', f, 'audio/wav')},
                   headers={'ngrok-skip-browser-warning': 'true'},
                   timeout=300)
    result = r.json()
    print(f'Shape: {result["shape"]}')
    print(f'Inference time: {result["inference_time_s"]}s')
    print('✅ Worker endpoint test passed!')

In [ ]:
# ============================================================
# STEP 8: Keep alive (leave this running)
# ============================================================
import time

print(f'Worker running at: {public_url}')
print('Press the stop button (or interrupt kernel) to shut down.\n')

while True:
    time.sleep(60)
    print(f'[{time.strftime("%H:%M:%S")}] Worker still running at {public_url}')